# Step10e — Dashboard Interactivo (versión Jupyter / ipywidgets)

**Proyecto:** HCI Ecuador (Hazard Chronicity Index)
**Fase:** 10 — Productos, sección D (continuación): versión editable del
dashboard, dentro de Jupyter (Fase 0 §9)
**Autor / Author:** Jerson Achicanoy, Information Management and GIS Expert
**Fecha:** 2026-08-06

**Objetivo de este notebook:** la misma lógica de filtrado del
dashboard HTML autónomo (`Step10d`), pero como widgets de Jupyter
(`ipywidgets`) — para que Jerson (o cualquiera con Jupyter) pueda abrir
este notebook, ajustar filtros con controles interactivos, y **editar
directamente el código** de cada gráfico si necesita una vista distinta
a la que ofrece el HTML estático. Es el complemento "editable" que el
HTML autónomo no puede ser por diseño.

**Requiere Jupyter corriendo** con `ipywidgets` instalado — a
diferencia de `HCI_Ecuador_Dashboard.html` (Step10d), no se puede
compartir como un archivo que cualquiera abra en su navegador sin
Jupyter.

**English:** Interactive dashboard for Jupyter (ipywidgets): same inputs and filters as the standalone HTML dashboard.

## 0. Configuración del entorno

**English:** Environment setup.

In [ ]:
# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger

# Tabular data
import pandas as pd
# Numerical arrays
import numpy as np
# Vector geodata
import geopandas as gpd
# Plotting library
import matplotlib.pyplot as plt
# Interactive widgets for Jupyter
import ipywidgets as widgets
# Display helper
from IPython.display import display

# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration
config = load_config(CONFIG_PATH)
# Create the notebook logger
logger = get_logger("step10e_dashboard_interactive", config, log_dir=PROJECT_ROOT / "logs")
# Log the start of the notebook
logger.info("Fase 10e iniciada — configuración cargada desde %s", CONFIG_PATH)

# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())

## 1. Datos (mismos insumos que el dashboard HTML de Step10d)

**English:** Data (same inputs as the HTML dashboard).

In [ ]:
# Read the master canton dataset
master = gpd.read_file(processed_dir / "mapa_master_canton.gpkg")

# Read the georeferenced events
he = gpd.read_file(processed_dir / "hazard_event_geo.gpkg", layer="hazard_event")
# Official canton and province names
canton_names = gpd.read_file(processed_dir / "adm2_canton.geojson")[["canton_code", "canton_name_official"]]
# Canton codes as strings
canton_names["canton_code"] = canton_names["canton_code"].astype(str)
# Canton codes as strings
he["canton_code"] = he["canton_code"].astype(str)
# Attach official names to each event
he = he.merge(canton_names, on="canton_code", how="left")

# Impact variables offered in the dashboard (column: label)
VARIABLES_IMPACTO = {
    "n_eventos": "Nº de Eventos",  # tratada aparte (conteo de filas, no una columna de suma)
    "affected_direct": "Afectados Directos",
    "affected_indirect": "Afectados Indirectos",
    "deaths": "Fallecidos",
    "houses_affected": "Viviendas Afectadas",
    "houses_destroyed": "Viviendas Destruidas",
    "crops_affected_ha": "Cultivos Afectados (ha)",
    "livestock_affected": "Ganado Afectado",
    "livestock_deaths": "Muertes de Ganado",
}

# Province options, with an 'all' entry
PROVINCIAS = ["(Todas)"] + sorted(master["province_name_official"].unique().tolist())
# ENSO category options, with an 'all' entry
CATEGORIAS_ENSO = ["(Todas)"] + sorted(he["icen_category"].dropna().unique().tolist())
# Years present in the data
ANIOS = sorted(he["year"].unique().tolist())
# Month options, with an 'all' entry
MESES = ["(Todos)"] + list(range(1, 13))

logger.info("Datos cargados: %d cantones, %d eventos", len(master), len(he))

## 2. Widgets de filtro

Los mismos 5 filtros del dashboard HTML (Fase 0 §9): Provincia,
Categoría ENSO, rango de Año, Mes, Variable de Impacto.

**English:** Filter widgets.

In [ ]:
# Province dropdown
w_provincia = widgets.Dropdown(options=PROVINCIAS, description="Provincia:", style={"description_width": "120px"})
# ENSO category dropdown
w_enso = widgets.Dropdown(options=CATEGORIAS_ENSO, description="Cat. ENSO:", style={"description_width": "120px"})
# Year range slider
w_anio = widgets.IntRangeSlider(value=[min(ANIOS), max(ANIOS)], min=min(ANIOS), max=max(ANIOS), step=1,
                                 description="Años:", style={"description_width": "120px"})
# Month dropdown
w_mes = widgets.Dropdown(options=MESES, description="Mes:", style={"description_width": "120px"})
# Impact variable dropdown
w_variable = widgets.Dropdown(options=list(VARIABLES_IMPACTO.items()), description="Variable:", style={"description_width": "120px"})
# Button that applies the filters
w_boton = widgets.Button(description="Aplicar filtros", button_style="primary")
# Output area for map, chart and table
salida = widgets.Output()

# Group the controls vertically
controles = widgets.VBox([w_provincia, w_enso, w_anio, w_mes, w_variable, w_boton])

## 3. Función de filtrado + redibujo (mapa, gráfico de tendencia, tabla)

**English:** Filtering and redraw function (map, annual chart, table).

In [ ]:
# Apply the current filters to the events
def filtrar_eventos():
    # Work on a copy
    df = he.copy()
    if w_provincia.value != "(Todas)":
        df = df[df["province_name_official"] == w_provincia.value]
    if w_enso.value != "(Todas)":
        df = df[df["icen_category"] == w_enso.value]
    df = df[(df["year"] >= w_anio.value[0]) & (df["year"] <= w_anio.value[1])]
    if w_mes.value != "(Todos)":
        df = df[df["month"] == w_mes.value]
    return df


# Redraw map, annual chart and top-20 table for the current filters
def redibujar(_=None):
    with salida:
        salida.clear_output(wait=True)
        variable_col, variable_label = w_variable.value, VARIABLES_IMPACTO[w_variable.value]
        df_filtrado = filtrar_eventos()

        if variable_col == "n_eventos":
            # Value per canton: event count or sum of the chosen variable
            por_canton = df_filtrado.groupby("canton_code").size().rename("valor").reset_index()
        else:
            por_canton = df_filtrado.groupby("canton_code")[variable_col].sum().rename("valor").reset_index()

        # Join values to the canton polygons
        gdf_plot = master.merge(por_canton, on="canton_code", how="left")
        # Cantons without events get 0
        gdf_plot["valor"] = gdf_plot["valor"].fillna(0)

        print(f"Filtro actual: Provincia={w_provincia.value} | ENSO={w_enso.value} | "
              f"Años={w_anio.value[0]}-{w_anio.value[1]} | Mes={w_mes.value} | Variable={variable_label}")
        print(f"Eventos en el filtro: {len(df_filtrado)} | Cantones con valor > 0: {(gdf_plot['valor'] > 0).sum()}")

        # Two panels: map and annual trend
        fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1.3, 1]})

        # Choropleth of the chosen variable
        gdf_plot.plot(column="valor", ax=axes[0], cmap="YlOrRd", legend=True,
                      edgecolor="white", linewidth=0.2, missing_kwds={"color": "lightgrey"})
        axes[0].set_title(f"{variable_label} — mapa cantonal (filtro actual)")
        axes[0].set_axis_off()

        # Annual totals for the chart
        por_anio = df_filtrado.groupby("year").size() if variable_col == "n_eventos" \
            else df_filtrado.groupby("year")[variable_col].sum()
        # Annual bars
        axes[1].bar(por_anio.index.astype(str), por_anio.values, color="#1f4e78")
        axes[1].set_title(f"{variable_label} — tendencia anual (filtro actual)")
        axes[1].tick_params(axis="x", rotation=45)

        plt.tight_layout()
        plt.show()

        # Top 20 cantons for the current filters
        top20 = gdf_plot.sort_values("valor", ascending=False).head(20)[
            ["canton_name_official", "province_name_official", "valor"]
        ].rename(columns={"canton_name_official": "Cantón", "province_name_official": "Provincia", "valor": variable_label})
        display(top20.reset_index(drop=True))


# Redraw when the button is pressed
w_boton.on_click(redibujar)

## 4. Mostrar el dashboard

**English:** Show the dashboard.

In [ ]:
# Show controls next to the output
display(widgets.HBox([controles, salida]))
# First render with default filters (whole period, no restrictions)
redibujar()  # primer render, con los filtros por defecto (todo el período, sin restricciones)

## 5. Notas para editar este dashboard

- Para agregar una variable nueva al filtro "Variable de Impacto":
  añadirla a `VARIABLES_IMPACTO` (Sección 1) — debe ser una columna
  numérica presente en `hazard_event_geo`.
- Para cambiar el tipo de mapa (p. ej. usar Jenks en vez del cmap
  continuo por defecto), editar la llamada a `gdf_plot.plot(...)` en
  `redibujar()` (Sección 3) — mismo patrón que Step10a.
- Este notebook no exporta ningún archivo por sí solo; es para
  exploración interactiva dentro de Jupyter. Para un archivo
  compartible, usar `Step10d_Ecuador_Dashboard_HTML.ipynb`.

**English:** Notes on how to edit this dashboard.